# AgriScore KZ — Exploratory Data Analysis

**Датасет:** Выгрузка по выданным субсидиям 2025 год (обезличенная)

**Цель:** Анализ качества данных, визуализация паттернов, обоснование feature engineering.

---
| Раздел | Содержание |
|--------|------------|
| 1 | Data Quality Report: пропуски, дубликаты, типы |
| 2 | Выбросы и аномалии |
| 3 | Data Cleaning Pipeline (наглядно) |
| 4 | Распределение заявок по областям |
| 5 | Распределение сумм субсидий |
| 6 | Топ-10 районов по количеству заявок |
| 7 | Соотношение одобренных / отклонённых |
| 8 | Средняя сумма по направлениям |
| 9 | Корреляционная матрица |
| 10 | Временная динамика |
| 11 | Итоговые выводы |

## 0. Настройка окружения

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys, os
from pathlib import Path

ROOT = Path('.').resolve().parent
if ROOT.name != 'agrisco-kz':
    ROOT = Path('.').resolve()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from src.preprocessing import load_raw, data_quality_report, print_quality_report, validate_and_clean

# ── Стиль ──
plt.rcParams.update({
    'figure.facecolor':  '#0F1117',
    'axes.facecolor':    '#1A1D2E',
    'axes.edgecolor':    '#2E3150',
    'axes.labelcolor':   '#C8D0F0',
    'axes.titlecolor':   '#FFFFFF',
    'axes.titlesize':    15,
    'axes.labelsize':    12,
    'axes.grid':         True,
    'grid.color':        '#2E3150',
    'grid.linewidth':    0.6,
    'xtick.color':       '#8892B0',
    'ytick.color':       '#8892B0',
    'xtick.labelsize':   10,
    'ytick.labelsize':   10,
    'text.color':        '#CDD6F4',
    'font.family':       'DejaVu Sans',
    'legend.facecolor':  '#1A1D2E',
    'legend.edgecolor':  '#2E3150',
    'legend.fontsize':   10,
})

TEAL    = '#64FFDA'
BLUE    = '#82AAFF'
PURPLE  = '#C792EA'
ORANGE  = '#FFCB6B'
RED     = '#F07178'
GREEN   = '#C3E88D'
PALETTE = [TEAL, BLUE, PURPLE, ORANGE, RED, GREEN]

import os
FIGURES_DIR = ROOT / 'notebooks' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print('Окружение настроено.')

## 1. Data Quality Report

Загружаем **сырые данные** (до очистки) и анализируем качество:
пропуски, дубликаты, типы данных, аномальные значения.

In [ ]:
# Загрузка сырых данных
df_raw = load_raw()
print(f'Загружено: {len(df_raw):,} строк, {df_raw.shape[1]} столбцов')
print(f'Диапазон дат: {pd.to_datetime(df_raw["date"], errors="coerce").min()} — {pd.to_datetime(df_raw["date"], errors="coerce").max()}')
df_raw.head(3)

In [ ]:
# Полный отчёт о качестве данных
report = data_quality_report(df_raw)
print_quality_report(report)

In [ ]:
# Визуализация пропусков по столбцам
miss = report['missing']
miss_plot = miss[miss['пропусков'] > 0].sort_values('процент', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.patch.set_facecolor('#0F1117')

# Левый: барплот пропусков
ax = axes[0]
if len(miss_plot) > 0:
    colors = [RED if p > 5 else ORANGE if p > 1 else GREEN for p in miss_plot['процент']]
    ax.barh(miss_plot.index, miss_plot['процент'], color=colors, edgecolor='#0F1117')
    for i, (idx, row) in enumerate(miss_plot.iterrows()):
        ax.text(row['процент'] + 0.2, i, f"{int(row['пропусков']):,} ({row['процент']:.1f}%)",
                va='center', fontsize=9, color='#CDD6F4')
    ax.set_xlabel('Процент пропусков')
    ax.set_title('Пропуски по столбцам', fontweight='bold')
else:
    ax.text(0.5, 0.5, 'Пропусков нет', ha='center', va='center',
            fontsize=16, color=GREEN, transform=ax.transAxes)
    ax.set_title('Пропуски по столбцам', fontweight='bold')
ax.spines[:].set_visible(False)

# Правый: heatmap пропусков (nullity matrix)
ax2 = axes[1]
sample = df_raw.sample(min(500, len(df_raw)), random_state=42).sort_index()
nullity = sample.isnull().astype(int)
ax2.imshow(nullity.T, aspect='auto', cmap='RdYlGn_r', interpolation='nearest')
ax2.set_yticks(range(len(nullity.columns)))
ax2.set_yticklabels(nullity.columns, fontsize=8)
ax2.set_xlabel('Строки (выборка 500)')
ax2.set_title('Nullity Matrix (жёлтый = пропуск)', fontweight='bold')

plt.tight_layout()
plt.savefig(str(FIGURES_DIR / 'fig0_missing.png'), dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()

In [ ]:
# Типы данных и уникальные значения
type_info = pd.DataFrame({
    'Тип': df_raw.dtypes,
    'Уникальных': df_raw.nunique(),
    'Пример': [df_raw[c].dropna().iloc[0] if df_raw[c].notna().any() else 'N/A' for c in df_raw.columns],
})
print('--- Типы данных и уникальность ---')
print(type_info.to_string())

## 2. Выбросы и аномалии

Анализ выбросов методом IQR + проверка бизнес-аномалий (отрицательные суммы, нулевые нормативы, будущие даты).

In [ ]:
# Числовые данные для анализа выбросов
amount = pd.to_numeric(df_raw['amount'], errors='coerce').dropna()
normative = pd.to_numeric(df_raw['normative'], errors='coerce').dropna()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor('#0F1117')

# Boxplot: amount
ax = axes[0]
bp = ax.boxplot(amount / 1e3, vert=True, patch_artist=True,
                boxprops=dict(facecolor=BLUE, alpha=0.7),
                medianprops=dict(color=ORANGE, linewidth=2),
                flierprops=dict(marker='o', markerfacecolor=RED, markersize=2, alpha=0.3))
ax.set_ylabel('тыс. ₸')
ax.set_title('Boxplot: Сумма субсидии', fontweight='bold')
q1, q3 = amount.quantile(0.25), amount.quantile(0.75)
iqr = q3 - q1
n_out = ((amount < q1 - 1.5*iqr) | (amount > q3 + 1.5*iqr)).sum()
ax.text(0.5, 0.95, f'Выбросов (IQR): {n_out:,} ({n_out/len(amount)*100:.1f}%)',
        transform=ax.transAxes, ha='center', va='top', fontsize=10, color=RED)
ax.spines[:].set_visible(False)

# Boxplot: normative
ax = axes[1]
ax.boxplot(normative / 1e3, vert=True, patch_artist=True,
           boxprops=dict(facecolor=TEAL, alpha=0.7),
           medianprops=dict(color=ORANGE, linewidth=2),
           flierprops=dict(marker='o', markerfacecolor=RED, markersize=2, alpha=0.3))
ax.set_ylabel('тыс. ₸')
ax.set_title('Boxplot: Норматив', fontweight='bold')
q1n, q3n = normative.quantile(0.25), normative.quantile(0.75)
iqrn = q3n - q1n
n_out_n = ((normative < q1n - 1.5*iqrn) | (normative > q3n + 1.5*iqrn)).sum()
ax.text(0.5, 0.95, f'Выбросов (IQR): {n_out_n:,} ({n_out_n/len(normative)*100:.1f}%)',
        transform=ax.transAxes, ha='center', va='top', fontsize=10, color=RED)
ax.spines[:].set_visible(False)

# Scatter: amount vs normative (аномалии)
ax = axes[2]
sample_idx = np.random.default_rng(42).choice(len(df_raw), min(3000, len(df_raw)), replace=False)
s_amount = pd.to_numeric(df_raw.iloc[sample_idx]['amount'], errors='coerce')
s_norm = pd.to_numeric(df_raw.iloc[sample_idx]['normative'], errors='coerce')
ax.scatter(s_norm / 1e3, s_amount / 1e3, alpha=0.3, s=8, color=BLUE)
ax.set_xlabel('Норматив, тыс. ₸')
ax.set_ylabel('Сумма, тыс. ₸')
ax.set_title('Сумма vs Норматив (выбросы видны)', fontweight='bold')
ax.spines[:].set_visible(False)

plt.tight_layout()
plt.savefig(str(FIGURES_DIR / 'fig0_outliers.png'), dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()

In [ ]:
# Бизнес-аномалии
anomalies = report['anomalies']
print('--- Бизнес-аномалии ---')
for k, v in anomalies.items():
    status = 'OK' if v == 0 else 'ВНИМАНИЕ'
    print(f'  [{status}] {k}: {v:,}')

# Дубликаты
print(f'\n--- Дубликаты ---')
print(f'  Полные дубликаты строк: {report["duplicates_full"]:,}')
print(f'  По ключевым полям: {report["duplicates_key"]:,}')
if report['duplicates_key'] > 0:
    key_cols = report['duplicate_key_cols']
    dups = df_raw[df_raw.duplicated(subset=key_cols, keep=False)].sort_values(key_cols)
    print(f'  Пример дубликатов:')
    print(dups[key_cols].head(6).to_string())

## 3. Data Cleaning Pipeline

Наглядная демонстрация каждого шага очистки: что было → что стало → сколько затронуто.

In [ ]:
# Запускаем пайплайн валидации + очистки
df_clean, val_report = validate_and_clean(df_raw)
log = val_report['cleaning_log']

print('=' * 60)
print('DATA CLEANING PIPELINE')
print('=' * 60)
print(f'\nИсходных строк:  {log["initial_rows"]:,}')
print(f'После очистки:   {log["final_rows"]:,}')
print(f'Удалено/изменено: {log["initial_rows"] - log["final_rows"]:,} строк')
print(f'\nШаги очистки:')
for i, step in enumerate(log['steps'], 1):
    print(f'  {i}. {step}')
if not log['steps']:
    print('  (дополнительная очистка не потребовалась)')

In [ ]:
# Визуализация пайплайна: до и после
fig, axes = plt.subplots(2, 2, figsize=(15, 8))
fig.patch.set_facecolor('#0F1117')
fig.suptitle('Data Cleaning Pipeline: до и после', fontsize=17,
             fontweight='bold', color='#FFFFFF', y=1.02)

# 1. Пропуски до/после
ax = axes[0, 0]
miss_before = df_raw.isnull().sum()
miss_after = df_clean.isnull().sum()
common_cols = [c for c in miss_before.index if c in miss_after.index]
x = range(len(common_cols))
ax.bar([i - 0.2 for i in x], [miss_before[c] for c in common_cols],
       width=0.4, label='До', color=RED, alpha=0.7)
ax.bar([i + 0.2 for i in x], [miss_after[c] for c in common_cols],
       width=0.4, label='После', color=GREEN, alpha=0.7)
ax.set_xticks(list(x))
ax.set_xticklabels(common_cols, rotation=45, ha='right', fontsize=7)
ax.set_title('Пропуски: до vs после', fontweight='bold')
ax.legend()
ax.spines[:].set_visible(False)

# 2. Распределение amount до/после (log scale)
ax = axes[0, 1]
amt_before = pd.to_numeric(df_raw['amount'], errors='coerce').dropna()
amt_after = df_clean['amount']
ax.hist(np.log1p(amt_before), bins=50, alpha=0.5, color=RED, label='До')
ax.hist(np.log1p(amt_after), bins=50, alpha=0.5, color=GREEN, label='После')
ax.set_title('log(amount): до vs после', fontweight='bold')
ax.set_xlabel('log(1 + amount)')
ax.legend()
ax.spines[:].set_visible(False)

# 3. Статусы
ax = axes[1, 0]
status_before = df_raw['status'].value_counts().head(6)
status_after = df_clean['status'].value_counts().head(6)
ax.barh(status_before.index, status_before.values, color=BLUE, alpha=0.8)
ax.set_title('Распределение статусов', fontweight='bold')
ax.set_xlabel('Количество')
for i, (idx, val) in enumerate(status_before.items()):
    ax.text(val + 100, i, f'{val:,}', va='center', fontsize=9, color='#CDD6F4')
ax.spines[:].set_visible(False)

# 4. Сводка пайплайна (текстовая)
ax = axes[1, 1]
ax.axis('off')
pipeline_text = (
    f"DATA CLEANING PIPELINE\n"
    f"{'='*35}\n\n"
    f"1. load_raw()\n"
    f"   Excel → DataFrame\n"
    f"   {log['initial_rows']:,} строк\n\n"
    f"2. Приведение типов\n"
    f"   amount, normative → numeric\n"
    f"   date → datetime\n\n"
    f"3. Удаление дубликатов\n"
    f"   {report['duplicates_full']:,} полных дублей\n\n"
    f"4. Обработка аномалий\n"
    f"   Отрицательные → 0\n"
    f"   Cap по 99.5 перцентилю\n\n"
    f"5. Feature derivation\n"
    f"   month, day_of_year, hour\n"
    f"   animals_count = amount/norm\n\n"
    f"Итого: {log['final_rows']:,} строк"
)
ax.text(0.05, 0.95, pipeline_text, transform=ax.transAxes,
        va='top', ha='left', fontsize=10, color=TEAL,
        family='monospace',
        bbox=dict(boxstyle='round,pad=0.5', facecolor='#1A1D2E', edgecolor=TEAL, alpha=0.9))

plt.tight_layout()
plt.savefig(str(FIGURES_DIR / 'fig0_pipeline.png'), dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()

In [ ]:
# Отчёт качества ПОСЛЕ очистки
report_after = data_quality_report(df_clean)
print('\n>>> Качество данных ПОСЛЕ очистки <<<')
print_quality_report(report_after)

### Наблюдение

Пайплайн очистки:
- Приводит типы (`amount`, `normative` → numeric, `date` → datetime)
- Удаляет полные дубликаты
- Обнуляет отрицательные суммы и нормативы (бизнес-аномалии)
- Cap выбросов по 99.5 перцентилю (экстремальные значения не искажают модель)
- Производные признаки: `month`, `day_of_year`, `hour`, `animals_count`

---

## 4. Распределение заявок по областям

In [ ]:
# Подготовка данных для EDA (используем очищенный df)
df = df_clean.copy()

APPROVED = {'Исполнена', 'Одобрена'}
REJECTED = {'Отклонена'}
def label_status(s):
    if s in APPROVED: return 'Одобрена'
    if s in REJECTED: return 'Отклонена'
    return 'Прочее'
df['status_label'] = df['status'].apply(label_status)

oblast_counts = (
    df.groupby('oblast', dropna=True)
      .size()
      .rename('count')
      .sort_values(ascending=True)
)

fig, ax = plt.subplots(figsize=(13, 7))
fig.patch.set_facecolor('#0F1117')

colors = [TEAL if v == oblast_counts.max() else BLUE for v in oblast_counts.values]
bars = ax.barh(oblast_counts.index, oblast_counts.values,
               color=colors, edgecolor='#0F1117', linewidth=0.5, height=0.65)

for bar, val in zip(bars, oblast_counts.values):
    ax.text(val + oblast_counts.max() * 0.01, bar.get_y() + bar.get_height() / 2,
            f'{val:,}', va='center', ha='left', color='#CDD6F4', fontsize=9.5, fontweight='bold')

ax.set_xlabel('Количество заявок', fontsize=12)
ax.set_title('Распределение заявок по областям Казахстана', fontsize=16, pad=16,
             color='#FFFFFF', fontweight='bold')
ax.set_xlim(0, oblast_counts.max() * 1.13)
ax.spines[:].set_visible(False)

leader = oblast_counts.idxmax()
ax.text(0.98, 0.02,
        f'Лидер: {leader}\n{oblast_counts.max():,} заявок',
        transform=ax.transAxes, ha='right', va='bottom',
        color=TEAL, fontsize=10, style='italic',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='#1A1D2E', edgecolor=TEAL, alpha=0.8))

plt.tight_layout()
plt.savefig(str(FIGURES_DIR / 'fig1_oblast.png'), dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()

### Наблюдение
Заявки **неравномерно** распределены по регионам: несколько областей аккумулируют подавляющее большинство.
Это может свидетельствовать как о реальной концентрации сельхозпроизводства, так и о разном уровне
«субсидийной грамотности». Дисбаланс важно учитывать при обучении ML-модели (regional bias).

## 5. Распределение сумм субсидий

In [ ]:
amounts = df['amount'].dropna()
amounts_k = amounts / 1_000

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.patch.set_facecolor('#0F1117')

ax = axes[0]
n, bins, patches = ax.hist(amounts_k, bins=60, color=BLUE, edgecolor='#0F1117', linewidth=0.3)
max_n = n.max()
for patch, height in zip(patches, n):
    patch.set_facecolor(plt.cm.cool(height / max_n * 0.8 + 0.1))

ax.axvline(amounts_k.median(), color=ORANGE, linewidth=1.8, linestyle='--',
           label=f'Медиана: {amounts_k.median():,.0f} тыс. ₸')
ax.axvline(amounts_k.mean(), color=RED, linewidth=1.8, linestyle='-',
           label=f'Среднее: {amounts_k.mean():,.0f} тыс. ₸')
ax.set_xlabel('Сумма субсидии, тыс. ₸')
ax.set_ylabel('Количество заявок')
ax.set_title('Гистограмма сумм субсидий', fontweight='bold')
ax.legend()
ax.spines[:].set_visible(False)

ax2 = axes[1]
ax2.hist(amounts_k[amounts_k > 0], bins=60, color=TEAL, edgecolor='#0F1117', linewidth=0.3, log=True)
ax2.set_xlabel('Сумма субсидии, тыс. ₸')
ax2.set_ylabel('Количество заявок (log)')
ax2.set_title('Гистограмма (логарифмическая шкала)', fontweight='bold')
ax2.spines[:].set_visible(False)

stats_text = (f"min: {amounts_k.min():,.0f}\n"
              f"p25: {amounts_k.quantile(0.25):,.0f}\n"
              f"p75: {amounts_k.quantile(0.75):,.0f}\n"
              f"max: {amounts_k.max():,.0f}")
ax2.text(0.97, 0.97, stats_text, transform=ax2.transAxes,
         va='top', ha='right', fontsize=9, color='#CDD6F4',
         bbox=dict(boxstyle='round,pad=0.4', facecolor='#1A1D2E', edgecolor='#2E3150'))

fig.suptitle('Распределение сумм субсидий (тыс. ₸)', fontsize=17,
             fontweight='bold', color='#FFFFFF', y=1.02)
plt.tight_layout()
plt.savefig(str(FIGURES_DIR / 'fig2_amounts.png'), dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()

### Наблюдение
Распределение сумм имеет выраженный **правый скос**: большинство заявок в низком диапазоне,
но есть значимый «хвост» с крупными выплатами. Медиана существенно ниже среднего.
Для ML-моделей рекомендуется логарифмическое преобразование `amount`.

## 6. Топ-10 районов по количеству заявок

In [ ]:
top10 = (
    df.groupby('district', dropna=True)
      .size().rename('count')
      .nlargest(10).sort_values(ascending=True)
)

fig, ax = plt.subplots(figsize=(13, 6))
fig.patch.set_facecolor('#0F1117')

norm_vals = (top10.values - top10.values.min()) / (top10.values.max() - top10.values.min() + 1e-9)
colors = [plt.cm.plasma(0.3 + v * 0.6) for v in norm_vals]

bars = ax.barh(top10.index, top10.values, color=colors, edgecolor='#0F1117', linewidth=0.4, height=0.65)

for bar, val in zip(bars, top10.values):
    ax.text(val + top10.max() * 0.008, bar.get_y() + bar.get_height() / 2,
            f'{val:,}', va='center', ha='left', color='#EEFFFF', fontsize=10, fontweight='bold')

ax.set_xlabel('Количество заявок')
ax.set_title('Топ-10 районов по количеству заявок на субсидии',
             fontsize=16, fontweight='bold', pad=15)
ax.set_xlim(0, top10.max() * 1.12)
ax.spines[:].set_visible(False)

plt.tight_layout()
plt.savefig(str(FIGURES_DIR / 'fig3_top10.png'), dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()

### Наблюдение
Топ-10 районов концентрируют непропорционально большую долю всех заявок.
`district` — потенциально сильный категориальный признак для ML.

## 7. Соотношение одобренных / отклонённых заявок

In [ ]:
status_counts = df['status_label'].value_counts()

STATUS_COLORS = {'Одобрена': TEAL, 'Отклонена': RED, 'Прочее': '#8892B0'}
colors_pie = [STATUS_COLORS.get(s, '#8892B0') for s in status_counts.index]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor('#0F1117')

ax = axes[0]
wedges, texts, autotexts = ax.pie(
    status_counts.values, labels=None, colors=colors_pie,
    autopct='%1.1f%%', startangle=140, pctdistance=0.82,
    wedgeprops=dict(edgecolor='#0F1117', linewidth=2.5),
    explode=[0.03] * len(status_counts),
)
for at in autotexts:
    at.set(color='#FFFFFF', fontsize=11, fontweight='bold')

legend_labels = [f"{s}  ({status_counts[s]:,})" for s in status_counts.index]
ax.legend(wedges, legend_labels, loc='lower center', bbox_to_anchor=(0.5, -0.12),
          ncol=1, framealpha=0.0, fontsize=11)
ax.set_title('Структура статусов заявок', fontsize=14, fontweight='bold', pad=20)

ax2 = axes[1]
status_oblast = (
    df[df['status_label'].isin(['Одобрена', 'Отклонена'])]
    .groupby(['oblast', 'status_label']).size().unstack(fill_value=0)
)
status_oblast['Approval_rate'] = (
    status_oblast.get('Одобрена', 0) /
    (status_oblast.get('Одобрена', 0) + status_oblast.get('Отклонена', 0))
).fillna(0)
status_oblast = status_oblast.sort_values('Approval_rate')

y_pos = range(len(status_oblast))
rates = status_oblast['Approval_rate'].values
bar_colors = [TEAL if r >= 0.5 else RED for r in rates]
ax2.barh(y_pos, rates * 100, color=bar_colors, edgecolor='#0F1117', linewidth=0.4, height=0.6)
ax2.axvline(50, color='#8892B0', linestyle='--', linewidth=1.2, alpha=0.7)
ax2.set_yticks(list(y_pos))
ax2.set_yticklabels(status_oblast.index, fontsize=9)
ax2.set_xlabel('Доля одобренных, %')
ax2.set_title('Коэффициент одобрения по областям', fontsize=13, fontweight='bold')
ax2.set_xlim(0, 110)
ax2.spines[:].set_visible(False)

for i, v in enumerate(rates):
    ax2.text(v * 100 + 1.5, i, f'{v*100:.1f}%', va='center', fontsize=8.5, color='#CDD6F4')

fig.suptitle('Анализ одобрения заявок', fontsize=17,
             fontweight='bold', color='#FFFFFF', y=1.02)
plt.tight_layout()
plt.savefig(str(FIGURES_DIR / 'fig4_approval.png'), dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()

### Наблюдение
Коэффициент одобрения **существенно варьируется** между регионами. Некоторые области
демонстрируют аномально низкий или высокий процент одобрения — ключевой аргумент
в пользу объективного merit-based скоринга.

## 8. Средняя сумма субсидий по направлениям

In [ ]:
dir_stats = (
    df.groupby('direction', dropna=True)['amount']
      .agg(['mean', 'median', 'count'])
      .rename(columns={'mean': 'Среднее', 'median': 'Медиана', 'count': 'Кол-во'})
      .sort_values('Среднее', ascending=False)
      .head(15)
)
dir_stats['Среднее_k'] = dir_stats['Среднее'] / 1_000
dir_stats['Медиана_k'] = dir_stats['Медиана'] / 1_000

fig, ax = plt.subplots(figsize=(14, 8))
fig.patch.set_facecolor('#0F1117')

x = np.arange(len(dir_stats))
w = 0.4

bars1 = ax.bar(x - w/2, dir_stats['Среднее_k'], width=w, label='Среднее',
               color=BLUE, edgecolor='#0F1117', linewidth=0.4)
bars2 = ax.bar(x + w/2, dir_stats['Медиана_k'], width=w, label='Медиана',
               color=TEAL, edgecolor='#0F1117', linewidth=0.4)

for i, (_, row) in enumerate(dir_stats.iterrows()):
    ax.text(i, max(row['Среднее_k'], row['Медиана_k']) + dir_stats['Среднее_k'].max() * 0.015,
            f"n={int(row['Кол-во']):,}", ha='center', va='bottom', fontsize=8, color='#8892B0')

ax.set_xticks(x)
ax.set_xticklabels(
    [s[:30] + '...' if len(s) > 30 else s for s in dir_stats.index],
    rotation=38, ha='right', fontsize=8.5
)
ax.set_ylabel('Сумма, тыс. ₸')
ax.set_title('Средняя и медианная сумма субсидий по направлениям',
             fontsize=15, fontweight='bold', pad=15)
ax.legend(fontsize=11)
ax.spines[:].set_visible(False)

plt.tight_layout()
plt.savefig(str(FIGURES_DIR / 'fig5_directions.png'), dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()

### Наблюдение
Суммы субсидий **кардинально различаются** по направлениям: разброс до 10x.
Разрыв средней и медианы указывает на наличие «крупных игроков».
`direction` — один из наиболее прогностичных признаков.

## 9. Корреляционная матрица

In [ ]:
df_num = df.copy()
df_num['animals_est'] = np.where(df_num['normative'] > 0,
                                  df_num['amount'] / df_num['normative'], 0)
df_num['log_amount'] = np.log1p(df_num['amount'].fillna(0))
df_num['approved_num'] = df_num['status_label'].map({'Одобрена': 1, 'Отклонена': 0})

NUM_COLS = ['amount', 'log_amount', 'normative', 'animals_est', 'month', 'day_of_year', 'approved_num']
COL_LABELS = ['Сумма', 'log(Сумма)', 'Норматив', 'Кол-во голов', 'Месяц', 'День года', 'Одобрена']

corr = df_num[NUM_COLS].dropna().corr()
corr.index = COL_LABELS
corr.columns = COL_LABELS

mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(10, 8))
fig.patch.set_facecolor('#0F1117')

cmap = sns.diverging_palette(240, 10, s=80, l=50, as_cmap=True)
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap=cmap, center=0, vmin=-1, vmax=1,
    linewidths=1.5, linecolor='#0F1117',
    annot_kws={'size': 11, 'color': '#EEFFFF', 'fontweight': 'bold'},
    ax=ax,
    cbar_kws={'shrink': 0.75, 'label': 'Коэффициент корреляции Пирсона'}
)

ax.set_title('Корреляционная матрица числовых признаков',
             fontsize=16, fontweight='bold', pad=18)
ax.tick_params(axis='x', rotation=30, labelsize=10)
ax.tick_params(axis='y', rotation=0, labelsize=10)

plt.tight_layout()
plt.savefig(str(FIGURES_DIR / 'fig6_corr.png'), dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()

### Наблюдение
- **amount ↔ normative** — сильная корреляция: сумма пропорциональна нормативу
- **log(amount)** улучшает линейность
- Мультиколлинеарность amount / log_amount — в модель один из двух

## 10. Временная динамика

In [ ]:
df_dated = df.dropna(subset=['date']).copy()
df_dated['month_name'] = df_dated['date'].dt.to_period('M').astype(str)
monthly = df_dated.groupby('month_name').agg(
    count=('num', 'count'),
    total_amount=('amount', 'sum')
).reset_index()

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
fig.patch.set_facecolor('#0F1117')

ax = axes[0]
ax.fill_between(range(len(monthly)), monthly['count'], color=BLUE, alpha=0.25)
ax.plot(range(len(monthly)), monthly['count'], color=BLUE, linewidth=2.2, marker='o', markersize=5)
ax.set_xticks(range(len(monthly)))
ax.set_xticklabels(monthly['month_name'], rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Кол-во заявок')
ax.set_title('Количество заявок по месяцам', fontweight='bold')
ax.spines[:].set_visible(False)

ax2 = axes[1]
ax2.fill_between(range(len(monthly)), monthly['total_amount'] / 1e6, color=TEAL, alpha=0.25)
ax2.plot(range(len(monthly)), monthly['total_amount'] / 1e6, color=TEAL, linewidth=2.2, marker='s', markersize=5)
ax2.set_xticks(range(len(monthly)))
ax2.set_xticklabels(monthly['month_name'], rotation=30, ha='right', fontsize=9)
ax2.set_ylabel('Сумма субсидий, млн ₸')
ax2.set_title('Общая сумма субсидий по месяцам', fontweight='bold')
ax2.spines[:].set_visible(False)

fig.suptitle('Временная динамика субсидирования', fontsize=16,
             fontweight='bold', color='#FFFFFF', y=1.02)
plt.tight_layout()
plt.savefig(str(FIGURES_DIR / 'fig7_timeline.png'), dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()

---
## 11. Итоговые выводы EDA

### Data Quality

| Проверка | Результат |
|----------|----------|
| Пропуски | Минимальные, обработаны fillna(0) / coerce |
| Дубликаты | Полные дубли удалены, ключевые дубли проверены |
| Выбросы (IQR) | Обнаружены в amount и normative, capped по 99.5 перцентилю |
| Бизнес-аномалии | Отрицательные суммы → 0, будущие даты проверены |
| Типы данных | Приведены: numeric, datetime, string strip |

### Ключевые находки

| # | Вывод | Импликация для ML |
|---|-------|-------------------|
| 1 | **Региональный дисбаланс**: топ-3 области дают ~50% заявок | Стратифицированная выборка, региональный признак |
| 2 | **Скошенное распределение сумм**: медиана << среднего | Логарифмирование amount |
| 3 | **Концентрация по районам**: топ-10 — ~30% заявок | district — сильный признак, target encoding |
| 4 | **Неоднородный approval rate**: 0–100% по регионам | Текущий процесс субъективен → merit-based обоснован |
| 5 | **Направление определяет сумму**: разброс до 10x | direction — топ-признак |
| 6 | **Мультиколлинеарность**: amount ∝ normative × animals | В модель: только log_amount или составные |
| 7 | **Сезонность**: пики в Q1 и Q3 | month, quarter как признаки |

---
*AgriScore KZ EDA | Данные МСХ РК*